In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np

raw_finance_data = {
    'Amount': [1250.50, 45.00, 300.00, 850.00, 12.99, 1500.00],
    'Vendor_ID': [101, 204, 315, 101, 204, 315],
    'Cost_Center': [5001, 5002, 5001, 5003, 5001, 5002],
    'GL_Account_Code': [0, 1, 2, 0, 1, 2] # 0: Travel, 1: Supplies, 2: Software
}

df = pd.DataFrame(raw_finance_data)

# Preprocessing: Normalize the numeric 'Amount' column
df['Amount_Norm'] = (df['Amount'] - df['Amount'].mean()) / df['Amount'].std()

# Extract features and targets as numpy arrays
features = df[['Amount_Norm', 'Vendor_ID', 'Cost_Center']].values
targets = df['GL_Account_Code'].values

# ------------------------------------------------------------------
# PYTORCH DATA CONVERSION
# ------------------------------------------------------------------
class LedgerDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

dataset = LedgerDataset(features, targets)
# Batch size of 2 simulates streaming batch updates
dataloader = DataLoader(dataset, batch_size=2, shuffle=True)

# ------------------------------------------------------------------
# CONSTRUCTING THE NEURAL AUDITOR
# ------------------------------------------------------------------
class AccountingClassifier(nn.Module):
    def __init__(self, input_dim=3, hidden_dim=8, num_classes=3):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, num_classes)
        )
    def forward(self, x):
        return self.network(x)

# Setup device (Colab offers free GPUs, but CPU works fine for this size)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = AccountingClassifier().to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.03)

# ------------------------------------------------------------------
# TRAINING LOOP
# ------------------------------------------------------------------
model.train()
for epoch in range(60):
    total_loss = 0
    for batch_X, batch_y in dataloader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)

        optimizer.zero_grad()
        predictions = model(batch_X)
        loss = criterion(predictions, batch_y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    if (epoch + 1) % 15 == 0:
        print(f"Epoch {epoch+1:02d}/60 | Batch Loss Average: {total_loss/len(dataloader):.4f}")

# ------------------------------------------------------------------
# LIVE INFERENCE (Predicting a new unmapped invoice)
# ------------------------------------------------------------------
model.eval()

# Let's say a new invoice comes in: $900.00 from Vendor 101 to Cost Center 5003
new_amount = 900.00
# We must normalize it exactly like our training data
new_amount_norm = (new_amount - df['Amount'].mean()) / df['Amount'].std()

# Format for the network: [Amount_Norm, Vendor_ID, Cost_Center]
new_tx = torch.tensor([[new_amount_norm, 101.0, 5003.0]], dtype=torch.float32).to(device)

with torch.no_grad():
    raw_outputs = model(new_tx)
    predicted_idx = torch.argmax(raw_outputs, dim=1).item()

gl_mapping = {0: "Travel Expense (GL-6100)", 1: "Office Supplies (GL-5200)", 2: "Software Licensing (GL-7400)"}
print("\n" + "="*50)
print(f"Colab Prediction Success!")
print(f"New Invoice Input: ${new_amount} from Vendor 101")
print(f"AI Suggested GL Account Routing -> {gl_mapping[predicted_idx]}")
print("="*50)